## Setup

In [65]:
import sys, os, re

import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(".."))

from config import PROCESSED_DATA_DIR, RAW_DATA_DIR, PROJECT_ROOT
from encoding import fix_mojibake_in_dataframe
from io_utils import read_json_utf8, write_csv_utf8

In [ ]:
PRODUCT_COLUMNS = [
    "product_id", "brand", "product_name", "product_url",
    "short_description", "long_description", "how_to_use",
    "ingredients", "loves_count", "hero_image_url", "size", 
]

SHADE_COLUMNS = [
    "sku_id", "product_id", "shade_name", "list_price", "sale_price",
    "in_stock", "is_final_sale", "badge", "swatch_img_url",
]

LEGACY_SHADE_COLUMN_ALIASES = {
    "image_url": "swatch_img_url",
}

UNIT_COLUMNS = ["oz_value", "g_value", "ml_value", "fl_oz_value"]

## Defined Functions

### Loading Data
Flattening JSON into 2 DataFrames, products and shades.

In [ ]:
# loading json file called "blush_detailed.json" and returns it as a JSON list of dict
def load_raw(filename: str ="blush_detailed.json") -> list[dict]:
    return read_json_utf8(RAW_DATA_DIR / filename)

In [ ]:
def build_products_table(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records) # turns the readed JSON list into DataFrame
    for col in PRODUCT_COLUMNS:
        if col not in df.columns:
            df[col] = None
    return df[PRODUCT_COLUMNS].drop_duplicates(subset="product_id")

In [24]:
df = build_products_table(load_raw())
df.head()

,product_id,brand,product_name,product_url,short_description,long_description,how_to_use,ingredients,loves_count,hero_image_url,size
0,P458747,PATRICK TA,Major Headlines Double-Take Crème & Powder Blu...,https://www.sephora.com/product/patrick-ta-maj...,A viral duo compact of powder and cream blush ...,What it is: A viral duo compact of powder and ...,Suggested Usage: -For a sheer look and dewy gl...,"Crème: PPG-3 Benzyl Ether Myristate, Phenyl Tr...",2651519,https://www.sephora.com/productimages/sku/s299...,0.17 oz crème and 0.17 oz powder/5 g crème and...
1,P97989778,Rare Beauty by Selena Gomez,Soft Pinch Liquid Blush,https://www.sephora.com/product/rare-beauty-by...,A weightless liquid blush that delivers high-i...,What it is: A weightless liquid blush that del...,Suggested Usage: -Gently remove excess product...,"Hydrogenated Polyisobutene, Hydrogenated Poly(...",3179039,https://www.sephora.com/productimages/sku/s291...,0.25 oz/7.5 mL
2,P515061,Saie,SuperSuede™ Radiant Talc-Free Baked Powder Blush,https://www.sephora.com/product/saie-supersued...,A long-wearing baked blush that’s non-comedoge...,What it is: A long-wearing baked blush that’s ...,"Suggested Usage: With a light hand, DIP flat s...","MA DONNA, BELLA, GRAZIE, CIAO MICA, SYNTHETIC ...",234325,https://www.sephora.com/productimages/sku/s297...,.01oz/3g
3,P517483,rhode,Pocket Blush Buildable Hydrating Cream Blush,https://www.sephora.com/product/pocket-blush-P...,An on-the-go cream blush that wakes up cheeks ...,What it is: An on-the-go cream blush that wake...,Suggested Usage: -Apply with your finger or a ...,"Octyldodecanol, Synthetic Wax, Hydrogenated Po...",1198021,https://www.sephora.com/productimages/sku/s299...,0.18 oz/5.3 g
4,P510756,HUDA BEAUTY,Blush Filter Soft Glow Liquid Blush,https://www.sephora.com/product/blush-filter-s...,A lightweight liquid blush with buildable pigm...,What it is: A lightweight liquid blush with bu...,Suggested Usage: -Add three dots of blush onto...,"Hydrogenated Polyisobutene, Hydrogenated Poly(...",419260,https://www.sephora.com/productimages/sku/s295...,0.15 oz/4.5 mL


In [28]:
## Flattening/Denormalizing `shades`
def build_shades_table(records: list[dict]) -> pd.DataFrame:
    rows = []
    for record in records:
        ## Looking for "shades" keys
        for shade in record.get("shades", []): # .get() if a product is missing the "shades" keys, it returns an empty list instead of crashing
            rows.append({**shade, "product_id": record.get("product_id")}) # ** takes all the key-value pairs inside "shades" dict and flatten them out
            ## then add "product_id" to the dict to track which shade belongs to which product_id
    df = pd.DataFrame(rows)
    df = df.rename(columns=LEGACY_SHADE_COLUMN_ALIASES)

    for col in SHADE_COLUMNS:
        if col not in df.columns:
            df[col] = None
            
    return df[SHADE_COLUMNS]

In [29]:
shades = build_shades_table(load_raw())
shades.head()

,sku_id,product_id,shade_name,list_price,sale_price,in_stock,is_final_sale,badge,swatch_img_url
0,2990125,P458747,Out Of Office,$40.00,None,False,False,NaN,https://www.sephora.com/productimages/sku/s299...
1,2742492,P458747,Just Enough,$40.00,None,False,False,NaN,https://www.sephora.com/productimages/sku/s274...
2,2742500,P458747,Not Too Much,$40.00,None,False,False,NaN,https://www.sephora.com/productimages/sku/s274...
3,2555894,P458747,She's Vibrant,$40.00,None,False,False,NaN,https://www.sephora.com/productimages/sku/s255...
4,2555886,P458747,She's a Doll,$40.00,None,False,False,NaN,https://www.sephora.com/productimages/sku/s255...


### Cleaning DataFrames

In [30]:
def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = fix_mojibake_in_dataframe(df)
    df = df.drop_duplicates(subset="product_id")
    return df

In [32]:
df = clean_products(df) # double cleaning for encoding problem
print(f"After Cleaning\n{df.info()}")
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   product_id         5 non-null      str  
 1   brand              5 non-null      str  
 2   product_name       5 non-null      str  
 3   product_url        5 non-null      str  
 4   short_description  5 non-null      str  
 5   long_description   5 non-null      str  
 6   how_to_use         5 non-null      str  
 7   ingredients        5 non-null      str  
 8   loves_count        5 non-null      int64
 9   hero_image_url     5 non-null      str  
 10  size               5 non-null      str  
dtypes: int64(1), str(10)
memory usage: 572.0 bytes
After Cleaning
None


,product_id,brand,product_name,product_url,short_description,long_description,how_to_use,ingredients,loves_count,hero_image_url,size
0,P458747,PATRICK TA,Major Headlines Double-Take Crème & Powder Blu...,https://www.sephora.com/product/patrick-ta-maj...,A viral duo compact of powder and cream blush ...,What it is: A viral duo compact of powder and ...,Suggested Usage: -For a sheer look and dewy gl...,"Crème: PPG-3 Benzyl Ether Myristate, Phenyl Tr...",2651519,https://www.sephora.com/productimages/sku/s299...,0.17 oz crème and 0.17 oz powder/5 g crème and...
1,P97989778,Rare Beauty by Selena Gomez,Soft Pinch Liquid Blush,https://www.sephora.com/product/rare-beauty-by...,A weightless liquid blush that delivers high-i...,What it is: A weightless liquid blush that del...,Suggested Usage: -Gently remove excess product...,"Hydrogenated Polyisobutene, Hydrogenated Poly(...",3179039,https://www.sephora.com/productimages/sku/s291...,0.25 oz/7.5 mL
2,P515061,Saie,SuperSuede™ Radiant Talc-Free Baked Powder Blush,https://www.sephora.com/product/saie-supersued...,A long-wearing baked blush that’s non-comedoge...,What it is: A long-wearing baked blush that’s ...,"Suggested Usage: With a light hand, DIP flat s...","MA DONNA, BELLA, GRAZIE, CIAO MICA, SYNTHETIC ...",234325,https://www.sephora.com/productimages/sku/s297...,.01oz/3g
3,P517483,rhode,Pocket Blush Buildable Hydrating Cream Blush,https://www.sephora.com/product/pocket-blush-P...,An on-the-go cream blush that wakes up cheeks ...,What it is: An on-the-go cream blush that wake...,Suggested Usage: -Apply with your finger or a ...,"Octyldodecanol, Synthetic Wax, Hydrogenated Po...",1198021,https://www.sephora.com/productimages/sku/s299...,0.18 oz/5.3 g
4,P510756,HUDA BEAUTY,Blush Filter Soft Glow Liquid Blush,https://www.sephora.com/product/blush-filter-s...,A lightweight liquid blush with buildable pigm...,What it is: A lightweight liquid blush with bu...,Suggested Usage: -Add three dots of blush onto...,"Hydrogenated Polyisobutene, Hydrogenated Poly(...",419260,https://www.sephora.com/productimages/sku/s295...,0.15 oz/4.5 mL


In [33]:
def clean_shades(df: pd.DataFrame) -> pd.DataFrame:
    df = fix_mojibake_in_dataframe(df)
    df = df.drop_duplicates(subset="product_id")
    return df

In [34]:
shades = clean_shades(shades)
shades.info()

<class 'pandas.DataFrame'>
Index: 5 entries, 0 to 60
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   sku_id          5 non-null      str   
 1   product_id      5 non-null      str   
 2   shade_name      5 non-null      str   
 3   list_price      5 non-null      str   
 4   sale_price      0 non-null      object
 5   in_stock        5 non-null      bool  
 6   is_final_sale   5 non-null      bool  
 7   badge           3 non-null      str   
 8   swatch_img_url  5 non-null      str   
dtypes: bool(2), object(1), str(6)
memory usage: 330.0+ bytes


### Handling Product Size
|Size|
|--|
|0.17 oz crème and 0.17 oz powder/5 g crème and 5 g powder|
|0.25 oz/7.5 mL|
|.01oz/3g|
|0.18 oz/5.3 g|
|0.15 oz/4.5 mL|

- Extract values from string

In [49]:
print("Current size")
df['size']

Current size


0    0.17 oz crème and 0.17 oz powder/5 g crème and...
1                                       0.25 oz/7.5 mL
2                                             .01oz/3g
3                                        0.18 oz/5.3 g
4                                       0.15 oz/4.5 mL
Name: size, dtype: str

In [46]:
def parse_size_units(size_str) -> pd.Series:
    if pd.isna(size_str): # check for missing values
        return pd.Series([np.nan]*4, index=UNIT_COLUMNS) # when the original spot is null, then all 4 cells are also null
    
    oz_val = g_val = ml_val = fl_oz_val = np.nan

    normalized = re.sub(r"fl\.?\s*oz", "fl_oz", str(size_str).lower().strip())

    for part in normalized.split("/"):
        for value_str, unit in re.findall(r"(\d*\.?\d+)\s*(oz|g|ml|fl_oz)\b", part):
            try:
                value = float(value_str)
            except ValueError:
                continue
            if unit == "oz" and pd.isna(oz_val):
                oz_val = value
            elif unit == "g" and pd.isna(g_val):
                g_val = value
            elif unit == "ml" and pd.isna(ml_val):
                ml_val = value
            elif unit == "fl_oz" and pd.isna(fl_oz_val):
                fl_oz_val = value
                
    return pd.Series([oz_val, g_val, ml_val, fl_oz_val], index=UNIT_COLUMNS)

In [54]:
def add_unit_columns(df: pd.DataFrame, size_column: str = "size") -> pd.DataFrame:
    """Add oz_value / g_value / ml_value / fl_oz_value columns derived from `size_column`."""
    df = df.copy()
    df[UNIT_COLUMNS] = df[size_column].apply(parse_size_units)
    df.drop("size", axis=1, inplace=True) # size, axis = 1: column, inplace
    return df

In [55]:
df = add_unit_columns(df)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   product_id         5 non-null      str    
 1   brand              5 non-null      str    
 2   product_name       5 non-null      str    
 3   product_url        5 non-null      str    
 4   short_description  5 non-null      str    
 5   long_description   5 non-null      str    
 6   how_to_use         5 non-null      str    
 7   ingredients        5 non-null      str    
 8   loves_count        5 non-null      int64  
 9   hero_image_url     5 non-null      str    
 10  oz_value           5 non-null      float64
 11  g_value            3 non-null      float64
 12  ml_value           2 non-null      float64
 13  fl_oz_value        0 non-null      float64
dtypes: float64(4), int64(1), str(9)
memory usage: 692.0 bytes


In [56]:
df.head()

,product_id,brand,product_name,product_url,short_description,long_description,how_to_use,ingredients,loves_count,hero_image_url,oz_value,g_value,ml_value,fl_oz_value
0,P458747,PATRICK TA,Major Headlines Double-Take Crème & Powder Blu...,https://www.sephora.com/product/patrick-ta-maj...,A viral duo compact of powder and cream blush ...,What it is: A viral duo compact of powder and ...,Suggested Usage: -For a sheer look and dewy gl...,"Crème: PPG-3 Benzyl Ether Myristate, Phenyl Tr...",2651519,https://www.sephora.com/productimages/sku/s299...,0.17,5.0,NaN,NaN
1,P97989778,Rare Beauty by Selena Gomez,Soft Pinch Liquid Blush,https://www.sephora.com/product/rare-beauty-by...,A weightless liquid blush that delivers high-i...,What it is: A weightless liquid blush that del...,Suggested Usage: -Gently remove excess product...,"Hydrogenated Polyisobutene, Hydrogenated Poly(...",3179039,https://www.sephora.com/productimages/sku/s291...,0.25,NaN,7.5,NaN
2,P515061,Saie,SuperSuede™ Radiant Talc-Free Baked Powder Blush,https://www.sephora.com/product/saie-supersued...,A long-wearing baked blush that’s non-comedoge...,What it is: A long-wearing baked blush that’s ...,"Suggested Usage: With a light hand, DIP flat s...","MA DONNA, BELLA, GRAZIE, CIAO MICA, SYNTHETIC ...",234325,https://www.sephora.com/productimages/sku/s297...,0.01,3.0,NaN,NaN
3,P517483,rhode,Pocket Blush Buildable Hydrating Cream Blush,https://www.sephora.com/product/pocket-blush-P...,An on-the-go cream blush that wakes up cheeks ...,What it is: An on-the-go cream blush that wake...,Suggested Usage: -Apply with your finger or a ...,"Octyldodecanol, Synthetic Wax, Hydrogenated Po...",1198021,https://www.sephora.com/productimages/sku/s299...,0.18,5.3,NaN,NaN
4,P510756,HUDA BEAUTY,Blush Filter Soft Glow Liquid Blush,https://www.sephora.com/product/blush-filter-s...,A lightweight liquid blush with buildable pigm...,What it is: A lightweight liquid blush with bu...,Suggested Usage: -Add three dots of blush onto...,"Hydrogenated Polyisobutene, Hydrogenated Poly(...",419260,https://www.sephora.com/productimages/sku/s295...,0.15,NaN,4.5,NaN


## Write and Export DataFrame

In [57]:
write_csv_utf8(df, PROCESSED_DATA_DIR / "df_1.csv")
write_csv_utf8(shades, PROCESSED_DATA_DIR / "shades_df_1.csv")

In [ ]:
print(f"Path of products: {PROJECT_ROOT}\df_1.csv")
print(f"Path of shades: {PROJECT_ROOT}\shades_df_1.csv")